In [2]:
!pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [3]:
from fastbook import *

In [4]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

### Language Model Using DataBlock

In [5]:
def get_imdb_sample(path):
    files = sorted(
        get_text_files(path, folders=['train', 'test', 'unsup'])
    )

    # Use a repeatable random sample of 10% of the reviews.
    sample_size = max(1, len(files) // 10)
    return random.Random(42).sample(files, sample_size)

get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

In [6]:
dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb_sample,
    splitter=RandomSplitter(0.1, seed=42)
).dataloaders(path, path=path, bs=50, seq_len=80)

In [7]:
dls_lm.show_batch(max_n=2)

,text,text_
0,"xxbos xxmaj this movie sucks . i hated it , every last minute that i allowed to waste my time . xxmaj it 's sorta nice to see that movies can still inspire sheer , honest dislike even after being taught to be accommodating about art . xxmaj but i think that this is where we draw the line , at least with xxmaj westerns . xxmaj this one was filmed in xxmaj isreal by xxmaj italians and with xxmaj","xxmaj this movie sucks . i hated it , every last minute that i allowed to waste my time . xxmaj it 's sorta nice to see that movies can still inspire sheer , honest dislike even after being taught to be accommodating about art . xxmaj but i think that this is where we draw the line , at least with xxmaj westerns . xxmaj this one was filmed in xxmaj isreal by xxmaj italians and with xxmaj american"
1,"is a woman with a bust that made xxmaj anita xxmaj ekberg look like xxmaj winona xxmaj ryder . xxmaj chesty was a stripper with an all natural 73 inch bust . xxmaj i 'm a fan of well endowed women , but that is just a bit too much for the average man 's liking i would suspect . xxmaj seeing her naked is an absolutely disgusting experience . xxmaj that s why the film is appealing however .","a woman with a bust that made xxmaj anita xxmaj ekberg look like xxmaj winona xxmaj ryder . xxmaj chesty was a stripper with an all natural 73 inch bust . xxmaj i 'm a fan of well endowed women , but that is just a bit too much for the average man 's liking i would suspect . xxmaj seeing her naked is an absolutely disgusting experience . xxmaj that s why the film is appealing however . xxmaj"


### Fine-Tuning the Language Model

In [8]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()]).to_fp16()

In [9]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,4.229225,4.137463,0.283509,62.643700,06:23


### Saving and Loading Models

In [10]:
learn.save('1epoch')

Path('/root/.fastai/data/imdb/models/1epoch.pth')

In [11]:
learn = learn.load('1epoch')

In [12]:
learn.unfreeze()
learn.fit_one_cycle(1, 2e-3) #10, 2e-3)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.813962,4.027203,0.296267,56.103748,07:01


In [13]:
learn.save_encoder('finetuned')


### Creating the Classifier DataLoaders

In [25]:
dls_clas = DataBlock(
    blocks=(TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
    get_y = parent_label,
    get_items=partial(get_text_files, folders=['train', 'test']),
    splitter=GrandparentSplitter(valid_name='test')
).dataloaders(path, path=path, bs=64, seq_len=72)

In [26]:
dls_clas.show_batch(max_n=3)

,text,category
0,"xxbos xxmaj match 1 : xxmaj tag xxmaj team xxmaj table xxmaj match xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley vs xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley started things off with a xxmaj tag xxmaj team xxmaj table xxmaj match against xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit . xxmaj according to the rules of the match , both opponents have to go through tables in order to get the win . xxmaj benoit and xxmaj guerrero heated up early on by taking turns hammering first xxmaj spike and then xxmaj bubba xxmaj ray . a xxmaj german xxunk by xxmaj benoit to xxmaj bubba took the wind out of the xxmaj dudley brother . xxmaj spike tried to help his brother , but the referee restrained him while xxmaj benoit and xxmaj guerrero",pos
1,"xxbos xxmaj some have praised xxunk xxmaj lost xxmaj xxunk as a xxmaj disney adventure for adults . i do n't think so -- at least not for thinking adults . \n\n xxmaj this script suggests a beginning as a live - action movie , that struck someone as the type of crap you can not sell to adults anymore . xxmaj the "" crack staff "" of many older adventure movies has been done well before , ( think xxmaj the xxmaj dirty xxmaj dozen ) but xxunk represents one of the worse films in that motif . xxmaj the characters are weak . xxmaj even the background that each member trots out seems stock and awkward at best . xxmaj an xxup md / xxmaj medicine xxmaj man , a tomboy mechanic whose father always wanted sons , if we have not at least seen these before ,",neg
2,"xxbos xxrep 3 * xxmaj warning - this review contains "" plot spoilers , "" though nothing could "" spoil "" this movie any more than it already is . xxmaj it really xxup is that bad . xxrep 3 * \n\n xxmaj before i begin , xxmaj i 'd like to let everyone know that this definitely is one of those so - incredibly - bad - that - you - fall - over - laughing movies . xxmaj if you 're in a lighthearted mood and need a very hearty laugh , this is the movie for you . xxmaj now without further ado , my review : \n\n xxmaj this movie was found in a bargain bin at wal - mart . xxmaj that should be the first clue as to how good of a movie it is . xxmaj secondly , it stars the lame action",neg


In [27]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, 
                                metrics=accuracy).to_fp16()

In [28]:
learn = learn.load_encoder('finetuned')

In [29]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.244366,0.200678,0.922720,03:49


In [30]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.217623,0.187048,0.929240,04:16


In [31]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.210745,0.178771,0.932440,05:52


In [32]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.166436,0.175665,0.934240,07:14
1,0.143101,0.175316,0.935560,07:14


In [ ]:
learn.save('1epoch')
learn.save_encoder('finetuned')

In [37]:

reviews = [
    "I loved this movie. The acting was wonderful.",
    "This movie was boring and a complete waste of time.",
    "The actors were excellent, but the story disappointed me.",
    "I didn't liked that show"
]

classes = list(learn.dls.vocab[1])

for review in reviews:
    label, class_index, probabilities = learn.predict(review)

    print("\nReview:", review)
    print("Predicted sentiment:", label)
    print(
        "Class probabilities:",
        {
            str(name): float(probability)
            for name, probability in zip(classes, probabilities)
        },
    )


Review: I loved this movie. The acting was wonderful.
Predicted sentiment: pos
Class probabilities: {'neg': 1.960230656550266e-05, 'pos': 0.9999804496765137}



Review: This movie was boring and a complete waste of time.
Predicted sentiment: neg
Class probabilities: {'neg': 0.9999645948410034, 'pos': 3.535625000949949e-05}



Review: The actors were excellent, but the story disappointed me.
Predicted sentiment: pos
Class probabilities: {'neg': 0.08567102253437042, 'pos': 0.9143289923667908}



Review: I didn't liked that show
Predicted sentiment: neg
Class probabilities: {'neg': 0.7884610295295715, 'pos': 0.21153894066810608}
